In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


df1=pd.read_csv('inventory.csv')
df2=pd.read_csv('sales.csv')

In [23]:
df1.isnull().sum()#0
df1.drop_duplicates(inplace=True)
df1.columns

Index(['Title', 'Author', 'Genre', 'Price', 'Quantity'], dtype='str')

In [19]:
df2.isnull().sum()#0
df2.drop_duplicates(inplace=True)
df2.columns
df2['Date']=pd.to_datetime(df2['Date'])
df2['Date'].astype

<bound method NDFrame.astype of 0     2026-01-02
1     2026-01-03
2     2026-01-05
3     2026-01-05
4     2026-01-07
         ...    
128   2026-09-20
129   2026-09-21
130   2026-09-22
131   2026-09-24
132   2026-09-26
Name: Date, Length: 133, dtype: datetime64[us]>

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime


class Bookstore:

    def __init__(self, inventory, sales):
        self.inventory = inventory
        self.sales = sales

    # 1. ADD BOOK
    def add_book(self):

        title = input("Enter title: ")
        author = input("Enter author: ")
        genre = input("Enter genre: ")
        price = float(input("Enter price: "))
        quantity = int(input("Enter quantity: "))

        if price <= 0:
            print("Price must be greater than 0")
            return

        if quantity < 0:
            print("Quantity cannot be negative")
            return

        if len(title) == 0:
            print("Title cannot be empty")
            return

        if title in self.inventory["Title"].values:
            print("Book already exists")
            return

        new_book = pd.DataFrame({
            "Title": [title],
            "Author": [author],
            "Genre": [genre],
            "Price": [price],
            "Quantity": [quantity]
        })

        self.inventory = pd.concat(
            [self.inventory, new_book],
            ignore_index=True
        )

        print("Book added successfully")


    # 2. REMOVE BOOK
    def remove_book(self):

        title = input("Enter title to remove: ")

        if title not in self.inventory["Title"].values:
            print("Book not found")
            return

        self.inventory = self.inventory[
            self.inventory["Title"] != title
        ].reset_index(drop=True)

        print("Book removed successfully")


    # 3. UPDATE INVENTORY
    def update_inventory(self):

        title = input("Enter title: ")

        if title not in self.inventory["Title"].values:
            print("Book not found")
            return

        quantity = int(input("Enter new quantity: "))

        if quantity < 0:
            print("Quantity cannot be negative")
            return

        self.inventory.loc[
            self.inventory["Title"] == title,
            "Quantity"
        ] = quantity

        print("Inventory updated successfully")


    # 4. RECORD SALE
    def record_sale(self):

        title = input("Enter title: ")

        if title not in self.inventory["Title"].values:
            print("Book not found")
            return

        quantity_sold = int(input("Enter quantity sold: "))

        if quantity_sold <= 0:
            print("Invalid quantity")
            return

        available_quantity = self.inventory.loc[
            self.inventory["Title"] == title,
            "Quantity"
        ].iloc[0]

        if quantity_sold > available_quantity:
            print("Not enough stock")
            return

        price = self.inventory.loc[
            self.inventory["Title"] == title,
            "Price"
        ].iloc[0]

        revenue = price * quantity_sold

        # Update inventory
        self.inventory.loc[
            self.inventory["Title"] == title,
            "Quantity"
        ] = available_quantity - quantity_sold

        # Create sale record
        date = datetime.now().strftime("%Y-%m-%d")

        new_sale = pd.DataFrame({
            "Date": [date],
            "Title": [title],
            "Quantity Sold": [quantity_sold],
            "Total Revenue": [revenue]
        })

        self.sales = pd.concat(
            [self.sales, new_sale],
            ignore_index=True
        )

        print("Sale recorded successfully")
        print("Revenue:", revenue)


    # 5. GENERATE REPORT
    def generate_report(self):

        print("\n========== BOOKSTORE REPORT ==========")

        total_books = len(self.inventory)

        total_stock = self.inventory["Quantity"].sum()

        total_sold = self.sales["Quantity Sold"].sum()

        total_revenue = self.sales["Total Revenue"].sum()

        average_price = self.inventory["Price"].mean()

        print("Total book titles:", total_books)
        print("Total stock:", total_stock)
        print("Total books sold:", total_sold)
        print("Total revenue:", total_revenue)
        print("Average book price:", average_price)

        if len(self.sales) > 0:

            best_seller = (
                self.sales
                .groupby("Title")["Quantity Sold"]
                .sum()
                .idxmax()
            )

            best_seller_quantity = (
                self.sales
                .groupby("Title")["Quantity Sold"]
                .sum()
                .max()
            )

            print("Best selling book:", best_seller)
            print("Books sold:", best_seller_quantity)

        print("======================================")


    # 6. NUMPY ANALYSIS
    def numpy_analysis(self):

        print("\n========== NUMPY ANALYSIS ==========")

        prices = self.inventory["Price"].to_numpy()

        sales = self.sales["Quantity Sold"].to_numpy()

        revenue = self.sales["Total Revenue"].to_numpy()

        if len(prices) > 0:
            print("Average price:", np.mean(prices))

        if len(sales) > 0:

            print("Total books sold:", np.sum(sales))

            print("Average books sold:", np.mean(sales))

        if len(revenue) > 0:

            print("Total revenue:", np.sum(revenue))

        print("====================================")


    # 7. SALES BY GENRE
    def sales_by_genre(self):

        data = self.sales.merge(
            self.inventory[["Title", "Genre"]],
            on="Title",
            how="left"
        )

        result = data.groupby("Genre")["Quantity Sold"].sum()

        print("\nSales by Genre:")
        print(result)

        return result


    # 8. SALES BY AUTHOR
    def sales_by_author(self):

        data = self.sales.merge(
            self.inventory[["Title", "Author"]],
            on="Title",
            how="left"
        )

        result = data.groupby("Author")["Quantity Sold"].sum()

        print("\nSales by Author:")
        print(result)

        return result


    # 9. BEST SELLING BOOK
    def best_selling_books(self):

        result = (
            self.sales
            .groupby("Title")["Quantity Sold"]
            .sum()
            .sort_values(ascending=False)
        )

        print("\nBest Selling Books:")
        print(result)

        return result


    # 10. REVENUE BY GENRE
    def revenue_by_genre(self):

        data = self.sales.merge(
            self.inventory[["Title", "Genre"]],
            on="Title",
            how="left"
        )

        result = data.groupby("Genre")["Total Revenue"].sum()

        print("\nRevenue by Genre:")
        print(result)

        return result


    # 11. BAR CHART
    def bar_chart(self):

        data = self.sales.merge(
            self.inventory[["Title", "Genre"]],
            on="Title",
            how="left"
        )

        result = data.groupby("Genre")["Quantity Sold"].sum()

        plt.figure(figsize=(8, 5))

        result.plot(kind="bar")

        plt.title("Sales by Genre")
        plt.xlabel("Genre")
        plt.ylabel("Quantity Sold")
        plt.xticks(rotation=45)

        plt.tight_layout()
        plt.show()


    # 12. LINE GRAPH
    def line_graph(self):

        data = self.sales.copy()

        data["Date"] = pd.to_datetime(data["Date"])

        monthly_sales = (
            data.groupby(
                data["Date"].dt.to_period("M")
            )["Total Revenue"]
            .sum()
        )

        plt.figure(figsize=(10, 5))

        plt.plot(
            monthly_sales.index.astype(str),
            monthly_sales.values,
            marker="o"
        )

        plt.title("Monthly Sales Trend")
        plt.xlabel("Month")
        plt.ylabel("Revenue")

        plt.xticks(rotation=45)

        plt.tight_layout()
        plt.show()


    # 13. PIE CHART
    def pie_chart(self):

        data = self.sales.merge(
            self.inventory[["Title", "Genre"]],
            on="Title",
            how="left"
        )

        revenue = data.groupby("Genre")["Total Revenue"].sum()

        plt.figure(figsize=(7, 7))

        plt.pie(
            revenue,
            labels=revenue.index,
            autopct="%1.1f%%"
        )

        plt.title("Revenue Share by Genre")

        plt.show()


    # 14. HEATMAP
    def heatmap(self):

        data = self.sales.merge(
            self.inventory[["Title", "Price"]],
            on="Title",
            how="left"
        )

        correlation = data[
            ["Price", "Quantity Sold"]
        ].corr()

        plt.figure(figsize=(6, 4))

        sns.heatmap(
            correlation,
            annot=True,
            cmap="coolwarm"
        )

        plt.title("Price and Sales Correlation")

        plt.show()


    # 15. SHOW INVENTORY
    def show_inventory(self):

        print("\n========== INVENTORY ==========")
        print(self.inventory)
        print("===============================")


    # 16. SHOW SALES
    def show_sales(self):

        print("\n========== SALES ==========")
        print(self.sales)
        print("===========================")


# -------------------------------------------------
# LOAD DATA
# -------------------------------------------------

df1 = pd.read_csv("inventory.csv")
df2 = pd.read_csv("sales.csv")


# Create object
bookstore = Bookstore(df1, df2)


# -------------------------------------------------
# MENU
# -------------------------------------------------

while True:

    print("\n========== BOOKSTORE SYSTEM ==========")

    print("1. Add Book")
    print("2. Remove Book")
    print("3. Update Inventory")
    print("4. Record Sale")
    print("5. Show Inventory")
    print("6. Show Sales")
    print("7. Generate Report")
    print("8. NumPy Analysis")
    print("9. Sales by Genre")
    print("10. Sales by Author")
    print("11. Best Selling Books")
    print("12. Revenue by Genre")
    print("13. Bar Chart")
    print("14. Line Graph")
    print("15. Pie Chart")
    print("16. Heatmap")
    print("17. Exit")

    choice = input("Enter your choice: ")


    if choice == "1":
        bookstore.add_book()

    elif choice == "2":
        bookstore.remove_book()

    elif choice == "3":
        bookstore.update_inventory()

    elif choice == "4":
        bookstore.record_sale()

    elif choice == "5":
        bookstore.show_inventory()

    elif choice == "6":
        bookstore.show_sales()

    elif choice == "7":
        bookstore.generate_report()

    elif choice == "8":
        bookstore.numpy_analysis()

    elif choice == "9":
        bookstore.sales_by_genre()

    elif choice == "10":
        bookstore.sales_by_author()

    elif choice == "11":
        bookstore.best_selling_books()

    elif choice == "12":
        bookstore.revenue_by_genre()

    elif choice == "13":
        bookstore.bar_chart()

    elif choice == "14":
        bookstore.line_graph()

    elif choice == "15":
        bookstore.pie_chart()

    elif choice == "16":
        bookstore.heatmap()

    elif choice == "17":
        print("Thank you for using Bookstore System")
        break

    else:
        print("Invalid choice")


========== BOOKSTORE SYSTEM ==========
1. Add Book
2. Remove Book
3. Update Inventory
4. Record Sale
5. Show Inventory
6. Show Sales
7. Generate Report
8. NumPy Analysis
9. Sales by Genre
10. Sales by Author
11. Best Selling Books
12. Revenue by Genre
13. Bar Chart
14. Line Graph
15. Pie Chart
16. Heatmap
17. Exit
Thank you for using Bookstore System
